In [1]:
import numpy as np
import pandas as pd
import random

user_col = "user_id"  
item_col = "app_id"

topk = 10
seed = 42

In [2]:

train_df = pd.read_csv("data/split/train_split.csv")
val_df = pd.read_csv("data/split/val_split.csv")
test_df = pd.read_csv("data/split/test_split.csv")

print("train:", train_df.shape, "users:", train_df[user_col].nunique())
print("val:  ", val_df.shape,   "users:", val_df[user_col].nunique())
print("test: ", test_df.shape,  "users:", test_df[user_col].nunique())


train: (63905, 8) users: 9906
val:   (11649, 8) users: 9906
test:  (12784, 8) users: 9906


In [3]:
steam = pd.read_csv("steam_store/steam.csv")
steam["genres"] = steam["genres"].fillna("")
steam["main_genre"] = steam["genres"].apply(
    lambda x: x.split(";")[0].strip() if isinstance(x, str) and x.strip() else "desconocido"
)
info_videojuegos = {
    row[item_col]: (row[item_col], row["main_genre"])
    for _, row in steam.iterrows()
}

In [ ]:
# todos los usuarios presentes en test
all_test_users = test_df[user_col].unique()

# relevantes segun is_recommended == True
test_rel_df = test_df[test_df["is_recommended"] == True]

rel_true_dict = (
    test_rel_df
    .groupby(user_col)[item_col]
    .apply(set)
    .to_dict()
)
# lo que se hace es construir user_rel_test para todos los usuarios,
# dejando set() si no tienen ningún True
user_rel_test = {
    uid: rel_true_dict.get(uid, set())
    for uid in all_test_users
}


In [5]:
def precision_at_k(rec_k, rel_set):
    if len(rec_k) == 0:
        return 0.0
    hits = sum((i in rel_set) for i in rec_k)
    return hits / len(rec_k)


def recall_at_k(rec_k, rel_set):
    if len(rel_set) == 0:
        return 0.0
    hits = sum((i in rel_set) for i in rec_k)
    return hits / len(rel_set)


def ndcg_at_k(rec_k, rel_set):
    if not rec_k:
        return 0.0
    dcg = 0.0
    for rank, it in enumerate(rec_k, start=1):
        if it in rel_set:
            dcg += 1.0 / np.log2(rank + 1)
    ideal = min(len(rel_set), len(rec_k))
    idcg = sum(1.0 / np.log2(i + 1) for i in range(1, ideal + 1))
    return (dcg / idcg) if idcg > 0 else 0.0


def hit_score_at_k(rec_k, rel_set):
    return 1.0 if any((i in rel_set) for i in rec_k) else 0.0


def map_at_k(rec_k, rel_set):
    if not rec_k:
        return 0.0
    ap_sum = 0.0
    hits = 0
    for rank, it in enumerate(rec_k, start=1):
        if it in rel_set:
            hits += 1
            ap_sum += hits / rank
    return ap_sum / len(rel_set) if len(rel_set) > 0 else 0.0


def diversity_at_k(user_topk, info_videojuegos):
    diversidades = []
    for uid, recs in user_topk.items():
        generos = {info_videojuegos[i][1] for i in recs if i in info_videojuegos}
        diversidades.append(len(generos))
    return float(np.mean(diversidades)) if diversidades else np.nan

In [6]:
def evaluate_from_topk(user_topk, user_rel, info_videojuegos=None, k=10, model_name=""):

    metrics_sum = {"precision": 0.0, "recall": 0.0,
                   "ndcg": 0.0, "hit": 0.0, "map": 0.0}
    n = 0

    user_topk_eval = {}

    for uid, rel_set in user_rel.items():
        uid_str = str(uid)
        if uid_str not in user_topk:
            continue

        rec_k = user_topk[uid_str][:k]
        user_topk_eval[uid_str] = rec_k

        metrics_sum["precision"] += precision_at_k(rec_k, rel_set)
        metrics_sum["recall"]    += recall_at_k(rec_k, rel_set)
        metrics_sum["ndcg"]      += ndcg_at_k(rec_k, rel_set)
        metrics_sum["hit"]       += hit_score_at_k(rec_k, rel_set)
        metrics_sum["map"]       += map_at_k(rec_k, rel_set)
        n += 1

    if n == 0:
        return pd.DataFrame([{
            "Modelo": model_name,
            f"Precision@{k}": np.nan,
            f"Recall@{k}":    np.nan,
            f"NDCG@{k}":      np.nan,
            f"F1-Score@{k}":  np.nan,
            f"HitRate@{k}":   np.nan,
            f"MAP@{k}":       np.nan,
            f"Diversity@{k}": np.nan,
            "Usuarios evaluados": 0
        }])

    mean_prec = metrics_sum["precision"] / n
    mean_rec  = metrics_sum["recall"]    / n
    mean_ndcg = metrics_sum["ndcg"]      / n
    mean_hit  = metrics_sum["hit"]       / n
    mean_map  = metrics_sum["map"]       / n

    f1 = 2 * mean_prec * mean_rec / (mean_prec + mean_rec + 1e-10)

    diversity = np.nan
    if info_videojuegos is not None and user_topk_eval:
        diversity = diversity_at_k(user_topk_eval, info_videojuegos)

    return pd.DataFrame([{
        "Modelo": model_name,
        f"Precision@{k}": mean_prec,
        f"Recall@{k}":    mean_rec,
        f"NDCG@{k}":      mean_ndcg,
        f"F1-Score@{k}":  f1,
        f"HitRate@{k}":   mean_hit,
        f"MAP@{k}":       mean_map,
        f"Diversity@{k}": diversity,
        "Usuarios evaluados": n
    }])


In [ ]:
def get_top_n_most_popular(train_df, n=10):

    item_popularity = (
        train_df
        .groupby(item_col)[user_col]
        .count()
        .sort_values(ascending=False)
    )
    popular_items = item_popularity.index.tolist()

    user_seen = (
        train_df
        .groupby(user_col)[item_col]
        .apply(set)
        .to_dict()
    )

    top_n = {}

    for uid, seen_items in user_seen.items():
        uid_str = str(uid)
        recs = []
        for iid in popular_items:
            if iid in seen_items:
                continue
            recs.append(iid)
            if len(recs) >= n:
                break
        top_n[uid_str] = recs

    return top_n


def evaluate_most_popular_metrics(top_n_most_popular, user_rel, info_videojuegos, k=10):

    return evaluate_from_topk(
        user_topk=top_n_most_popular,
        user_rel=user_rel,
        info_videojuegos=info_videojuegos,
        k=k,
        model_name="MostPopular"
    )



In [ ]:
def get_top_n_random(train_df, n=10, seed=42):

    random.seed(seed)
    all_items = set(train_df[item_col].unique())
    user_seen = (
        train_df
        .groupby(user_col)[item_col]
        .apply(set)
        .to_dict()
    )

    top_n = {}

    for uid in user_seen.keys():
        uid_str = str(uid)
        seen_items = user_seen[uid]
        available = list(all_items - seen_items)

        if len(available) == 0:
            top_n[uid_str] = []
            continue

        sampled = random.sample(available, min(n, len(available)))
        top_n[uid_str] = sampled

    return top_n


def evaluate_random_metrics(top_n_random, user_rel, info_videojuegos, k=10):

    return evaluate_from_topk(
        user_topk=top_n_random,
        user_rel=user_rel,
        info_videojuegos=info_videojuegos,
        k=k,
        model_name="Random"
    )

In [9]:
top_n_most_popular = get_top_n_most_popular(train_df, n=topk)
metrics_most_pop = evaluate_most_popular_metrics(
    top_n_most_popular,
    user_rel=user_rel_test,
    info_videojuegos=info_videojuegos,
    k=topk
)

top_n_random = get_top_n_random(train_df, n=topk, seed=seed)
metrics_random = evaluate_random_metrics(
    top_n_random,
    user_rel=user_rel_test,
    info_videojuegos=info_videojuegos,
    k=topk
)

In [10]:
metrics_most_pop

,Modelo,Precision@10,Recall@10,NDCG@10,F1-Score@10,HitRate@10,MAP@10,Diversity@10,Usuarios evaluados
0,MostPopular,0.003755,0.031476,0.016251,0.00671,0.03725,0.010862,2.966384,9906


In [11]:
metrics_random

,Modelo,Precision@10,Recall@10,NDCG@10,F1-Score@10,HitRate@10,MAP@10,Diversity@10,Usuarios evaluados
0,Random,0.000363,0.002934,0.001623,0.000647,0.003634,0.00113,3.441349,9906
